# Avvaiyar Paatti: run evals on Colab / Kaggle
1. On your Mac: zip the `v2` folder as `v2.zip` (include `eval/results_pipeline_*.jsonl` to resume; leave out `.env`, `.venv`, `.cache`).
2. Upload `v2.zip` here (Colab: Files panel; Kaggle: add it as a dataset).
3. Add your Gemini key as a secret named `GOOGLE_API_KEY` (Colab: key icon in the sidebar; Kaggle: Add-ons > Secrets).
4. Run the cells top to bottom. Runtime: GPU (T4) for the embedding benchmark; CPU is fine for the Gemini eval.

In [ ]:
import os, glob
z = (glob.glob('/content/v2.zip') + glob.glob('/kaggle/input/*/v2.zip') + glob.glob('/kaggle/input/*/*/v2.zip'))[0]
!unzip -q -o {z} -d /tmp/paatti
root = os.path.dirname(glob.glob('/tmp/paatti/**/paatti/engine.py', recursive=True)[0]) + '/..'
os.chdir(os.path.abspath(root)); print(os.getcwd())
!pip -q install -r requirements-eval.txt

In [ ]:
# Gemini key from the notebook's secret store (never paste it into a cell)
try:
    from google.colab import userdata; os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except ImportError:
    from kaggle_secrets import UserSecretsClient; os.environ['GOOGLE_API_KEY'] = UserSecretsClient().get_secret('GOOGLE_API_KEY')
print('key loaded:', bool(os.environ.get('GOOGLE_API_KEY')))

## A. Gemini end-to-end eval (resumes automatically if a results file was in the zip)

In [ ]:
import glob
prev = sorted(glob.glob('eval/results_pipeline_*.jsonl'))
resume = f'--resume {prev[-1]}' if prev else ''
print('resuming from', prev[-1] if prev else 'nothing (fresh run)')
!python eval/run_eval.py pipeline --embedder bge-m3 --judge {resume}

In [ ]:
# download the results file to keep it
f = sorted(glob.glob('eval/results_pipeline_*.jsonl'))[-1]; print(f)
try:
    from google.colab import files; files.download(f)
except ImportError:
    import shutil; shutil.copy(f, '/kaggle/working/'); print('saved to /kaggle/working (Output tab)')

## B. Raw embedding benchmark (optional: the 4 models your Mac couldn't download)

In [ ]:
!python eval/bench_embedders.py --models gte-ml e5-large-instruct arctic-l2 qwen3-0.6b bge-m3 mpnet-ml

In [ ]:
from IPython.display import Markdown
Markdown(open('eval/results_embedders.md').read())